# Fase 1 – Modelo predictivo de precios de viviendas

Proyecto Integrador v1 – 2026 II · Modelos y Simulación de Sistemas I

## 6.1 Introducción

- **Problema seleccionado:** estimar el precio de venta de una vivienda a partir de sus características físicas, de ubicación y de la transacción.
- **Objetivo del modelo:** dada la ficha de una vivienda, predecir su precio de venta en dólares (`SalePrice`). Esta fase busca un proceso **reproducible, documentado y sin fuga de información**, más que el mejor desempeño posible.
- **Tipo de problema:** aprendizaje supervisado – **regresión**.
- **Variable objetivo:** `SalePrice` (precio de venta, USD).
- **Fuente de los datos:** competencia de Kaggle [`aecincode_houseprices`](https://www.kaggle.com/competitions/aecincode_houseprices), basada en el *Ames Housing Dataset* (De Cock, 2011) y relacionada con [House Prices – Advanced Regression Techniques](https://www.kaggle.com/competitions/house-prices-advanced-regression-techniques/). Se usa `data/train.csv` (con precio) para entrenar y evaluar, y `data/test.csv` (sin precio) como "datos nuevos" para probar el modelo guardado.
- **Referencias metodológicas:** notebooks de clase NOTES 04.01 (exploración), 04.02 (limpieza y valores faltantes) y 04.03/04.04 (pruebas de hipótesis).

In [1]:
import os, json, platform, warnings
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import joblib
import sklearn

from sklearn.model_selection import train_test_split, KFold, cross_validate
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.inspection import permutation_importance
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

warnings.filterwarnings("ignore")
%matplotlib inline
sns.set()

# ---- configuración global (todo el notebook depende solo de estas constantes) ----
RANDOM_STATE = 42                 # semilla única para split, validación cruzada y modelo
np.random.seed(RANDOM_STATE)
DATA_PATH = "data/train.csv"      # datos con variable objetivo
NEW_DATA_PATH = "data/test.csv"   # datos nuevos (sin variable objetivo)
TARGET = "SalePrice"
IDS = ["Order", "PID", "Id"]      # identificadores: no describen la vivienda
TEST_SIZE = 0.2
MODEL_PATH = "modelo.joblib"

## 6.2 Descripción del conjunto de datos

In [2]:
d = pd.read_csv(DATA_PATH)
predictoras = [c for c in d.columns if c not in IDS + [TARGET]]
print(f"Observaciones: {len(d):,}")
print(f"Columnas: {d.shape[1]} = 1 objetivo + {len(predictoras)} predictoras + {len(IDS)} identificadores")

tipos = d[predictoras].dtypes.map(lambda t: "numérica" if pd.api.types.is_numeric_dtype(t) else "categórica")
print("\nTipos de predictoras:"); print(tipos.value_counts().to_string())
d[TARGET].describe().round(0).to_frame("SalePrice (USD)").T

Observaciones: 1,758
Columnas: 83 = 1 objetivo + 79 predictoras + 3 identificadores

Tipos de predictoras:
categórica    43
numérica      36


,count,mean,std,min,25%,50%,75%,max
SalePrice (USD),1758.0,179194.0,77798.0,12789.0,129425.0,160100.0,210000.0,755000.0


- **Observaciones:** cada fila es la venta de una vivienda distinta. **Variables:** 79 predictoras (36 numéricas y 43 categóricas; `MS SubClass` figura como numérica pero codifica categorías) más el objetivo y tres identificadores (`Order`, `PID`, `Id`) que solo sirven para identificar filas y se excluyen.
- **Variable objetivo:** `SalePrice`, precio de venta en dólares (continua, positiva).
- **Variables predictoras:** dimensiones y áreas (lote, sótano, garaje, pisos), calidad y condición de materiales y acabados (escalas Ex/Gd/TA/Fa/Po), año de construcción y remodelación, barrio, tipo de vivienda y condiciones de la venta.
- **Limitaciones:**
  - Los datos corresponden a una sola ciudad (Ames, Iowa) y a ventas de 2006–2010, por lo que el modelo no generaliza necesariamente a otros mercados ni épocas.
  - Aunque hay una columna de año de venta, **no es un problema de series de tiempo**: cada venta se trata como una observación independiente.
  - Muchos valores vacíos no son datos perdidos sino "la vivienda no tiene esa característica" (se analiza en 6.3 y 6.4).
  - Hay pocos valores atípicos muy influyentes (p. ej. viviendas de más de 4.000 pies² de área habitable).